In [1]:
from elasticsearch import Elasticsearch
import pandas as pd
import json
from ipywidgets import interact, Dropdown
import matplotlib.pyplot as plt
import warnings
import urllib3
import time
import requests
import numpy as np
from IPython.display import display
from sklearn.preprocessing import MinMaxScaler
import ipywidgets as widgets

# Suppress all warnings
warnings.filterwarnings("ignore")

# Suppress only InsecureRequestWarning from urllib3 (for verify_certs=False)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# === API Parameter ===
api_url = "http://localhost:9090/api/youtubelifeapi"

# === Define two ranges to avoid 10,000 limit ===
ranges = [
    {"start": "2025-03-01", "end": "2025-04-10"},
    {"start": "2025-04-11", "end": "2026-01-01"}
]

# === Fetch & Combine ===
all_data = []

for i, params in enumerate(ranges):
    response = requests.get(api_url, params=params)
    if response.status_code == 200:
        json_data = response.json()
        results = json_data.get("data", [])
        print(f"[OK] Retrieved {len(results)} documents from range {i+1}")
        all_data.append(pd.DataFrame(results))
    else:
        print(f"[X] Request {i+1} failed with status code {response.status_code}")

# === Combine & Clean ===
if all_data:
    ytl_df_raw = pd.concat(all_data, ignore_index=True)
    print(f"[✓] Combined total: {len(ytl_df_raw)} rows.")
    display(ytl_df_raw.columns)
else:
    print("[X] No data retrieved from any range.")

In [3]:
# Normalize data frame
df = pd.json_normalize(df_raw.to_dict(orient='records'))

# Ensure all relevant fields are numeric
df['statistics.viewCount'] = df['statistics.viewCount'].astype(float)
df['statistics.likeCount'] = df['statistics.likeCount'].astype(float)
df['statistics.commentCount'] = df['statistics.commentCount'].astype(float)
df['statistics.favoriteCount'] = df['statistics.favoriteCount'].astype(float)
df['snippet.publishedAt'] = pd.to_datetime(df['snippet.publishedAt'], utc=True)

# Convert to Melbourne time (UTC+10)
df['published_melbourne'] = df['snippet.publishedAt'].dt.tz_convert('Australia/Melbourne')

# Get current timestamp
now_ts = time.time()

# Convert published date to timestamp
df['published_timestamp'] = pd.to_datetime(df['snippet.publishedAt']).astype(np.int64) // 10**9

# Calculate hours since video was published
df['hours_since_post'] = (now_ts - df['published_timestamp']) / 3600

# Compute raw weighted score
df['raw_score'] = (
    df['statistics.viewCount'] * 0.5 +
    df['statistics.likeCount'] * 1.0 +
    df['statistics.commentCount'] * 1.5 +
    df['statistics.favoriteCount'] * 1.2
)

# Apply exponential decay based on how long ago it was published
df['decay'] = np.exp(-0.1 * df['hours_since_post'])

# Final decayed score (hotness score)
df['score1'] = df['raw_score'] * df['decay']

# Replace 0 in viewCount with 1 to avoid division by zero
df['statistics.viewCount'] = df['statistics.viewCount'].replace(0, 1)
df['statistics.likeCount'] = df['statistics.likeCount'].replace(0, 1)

# Compute engagement ratio score (likes + 2*comments + 3*favorites) per view
df['score2'] = (
    ((df['statistics.likeCount'] / df['statistics.viewCount']) +
    2 * (df['statistics.commentCount'] / df['statistics.viewCount']) +
    3 * (df['statistics.favoriteCount'] / df['statistics.viewCount'])) * 100
)

# Handle division by zero or missing values
df['score2'] = df['score2'].replace([np.inf, -np.inf], np.nan).fillna(0)

# Scale score1 and score2 to 1–100 range
df['statistics.viewCount'] = df['statistics.viewCount'].replace(0, np.nan)
df['score1'] = df['score1'].replace([np.inf, -np.inf], np.nan).fillna(0)
df['score2'] = df['score2'].replace([np.inf, -np.inf], np.nan).fillna(0)
scaler = MinMaxScaler(feature_range=(1, 100))
df['score1_scaled'] = scaler.fit_transform(df[['score1']])
df['score2_scaled'] = scaler.fit_transform(df[['score2']])

print(df.columns.tolist())
df.head()

['search_prompt', 'kind', 'etag', 'id', 'snippet.publishedAt', 'snippet.description', 'snippet.title', 'snippet.channelId', 'snippet.categoryId', 'snippet.channelTitle', 'snippet.liveBroadcastContent', 'contentDetails.duration', 'contentDetails.definition', 'contentDetails.dimension', 'statistics.likeCount', 'statistics.viewCount', 'statistics.favoriteCount', 'statistics.commentCount', 'snippet.tags', 'published_melbourne', 'published_timestamp', 'hours_since_post', 'raw_score', 'decay', 'score1', 'score2', 'score1_scaled', 'score2_scaled']


,search_prompt,kind,etag,id,snippet.publishedAt,snippet.description,snippet.title,snippet.channelId,snippet.categoryId,snippet.channelTitle,...,snippet.tags,published_melbourne,published_timestamp,hours_since_post,raw_score,decay,score1,score2,score1_scaled,score2_scaled
0,australia election,youtube#video,y8oTgEke_FNIn5oJHG0ZnkVXjQ8,-s48d6v_--c,2025-01-01 01:27:45+00:00,About this video📸📸\nBumrah vs sem konstas\nWHO...,BUMRAH🥎VS SAM CONSTAS VOTING CHALLENGE #cricke...,UCre_Lemuz_Z2dqY3-umEApw,17,The creative boys,...,NaN,2025-01-01 12:27:45+11:00,1735694865,3193.590162,68324.0,2.014381e-139,1.376305e-134,2.864128,1.0,1.405070
1,australia election,youtube#video,adXTmHIWzqVEYww1dGikjGB4Z-c,SHRcMkX_7Bk,2025-01-01 06:00:00+00:00,"Squid Game is now playing, only on Netflix: ht...",[Reaction] Squid Game Season 2 cast reacts to ...,UCpiCK8c6PBktcxq7Az_t4RQ,24,Netflix K-Content,...,"[JoYuri, KangHaneul, LeeByunghun, LeeJungjae, ...",2025-01-01 17:00:00+11:00,1735711200,3189.052662,1768595.5,3.171047e-139,5.608299e-133,1.945491,1.0,1.275148
2,australia election,youtube#video,6eYg4xRFcZDcg0ulqLxDHoyYAXY,9IWctwOLWMU,2025-01-01 07:39:18+00:00,Ind vs Aus: India's defeat in the Melbourne Te...,Exclusive: Melbourne Test Sparks Dressing Room...,UCJEDFSxHHOW1PpBccdSxOTA,25,The Indian Express,...,"[ind vs aus, ind vs aus test, ind vs aus test ...",2025-01-01 18:39:18+11:00,1735717158,3187.397662,32161.0,3.741781e-139,1.203394e-134,1.105016,1.0,1.156281
3,australia election,youtube#video,VgYPBpcZuoIZmtJ7U5DCkLcVXn0,b1jJrHhHR6k,2025-01-01 07:44:31+00:00,Sky News host James Macpherson says it is “no ...,‘No secret’: Australians face ‘major issues’ i...,UCO0akufu9MOzyz3nvGIXAAw,25,Sky News Australia,...,"[6366594032112, fb, msn, opinion, yt]",2025-01-01 18:44:31+11:00,1735717471,3187.310717,3992.5,3.774456e-139,1.506951e-135,4.894915,1.0,1.692281
4,australia election,youtube#video,69tn3ovbA3Mb881RIRv0MfjoONc,lib5XzgliLg,2025-01-01 09:28:16+00:00,,Horsley Electrical Wholesalers #election #ai #...,UC_RcaLYGWRh5wmBodMAdISA,22,Horsley electrical,...,NaN,2025-01-01 20:28:16+11:00,1735723696,3185.581551,26.5,4.486949e-139,1.189041e-137,1.960784,1.0,1.277311
